# Week 2 — Cosine Similarity & Nearest-Neighbour Search

**Phase 01: Embeddings & Semantic Similarity for Pension Fund Analytics**

---

## Recap from Week 1

Last week we:
- Loaded `all-MiniLM-L6-v2` and encoded a sentence into a 384-dimensional vector
- Demonstrated that `"coverage ratio"` and `"dekkingsgraad"` produce *close* vectors
- Embedded all 500 pension research abstracts in one batch
- Saved the embedding matrix to `article_embeddings.npy`

This week we build the *search* layer on top of those embeddings:

```
query string
     │
     ▼  model.encode()
query vector  ──── cosine similarity ──▶  ranked list of articles
     │                    ▲
     │          corpus embedding matrix
     │          (loaded from disk)
```

**New this week:** understand *why* cosine similarity works, implement it from scratch, compare keyword search vs. semantic search, and wrap everything in a reusable function.

In [ ]:
# ── Cell 1: Load embeddings and metadata from Week 1 ─────────────────────────
import numpy as np
import pandas as pd
import os
import time

EMBED_PATH = 'article_embeddings.npy'
META_PATH  = 'article_index_meta.csv'

if os.path.exists(EMBED_PATH) and os.path.exists(META_PATH):
    embeddings = np.load(EMBED_PATH)
    df = pd.read_csv(META_PATH)
    print(f'Loaded embeddings : {embeddings.shape}  (articles x dims)')
    print(f'Loaded metadata   : {df.shape}')
else:
    print('Embedding files not found. Re-building from scratch...')
    print('(Run week1_intro_embeddings.ipynb first to generate the files.)')
    print('Falling back to synthetic dataset for demonstration...')
    from sentence_transformers import SentenceTransformer
    DATA_PATH = os.path.join('..', 'data', 'raw', 'articles.csv')
    if os.path.exists(DATA_PATH):
        df = pd.read_csv(DATA_PATH)
    else:
        SYNTHETIC_DATA = [
            {'id': i, 'title': t, 'abstract': a, 'category': c, 'year': y, 'authors': 'Author', 'source_journal': 'Journal', 'word_count': 180}
            for i, (t, a, c, y) in enumerate([
                ('FTK Coverage Ratio', 'Dutch FTK requires pension funds to maintain a coverage ratio above 110%. Funds below the minimum required coverage ratio of 90% face regulatory action by DNB.', 'pension_regulation', 2022),
                ('ESG in ALM', 'ESG integration in asset-liability management for European pension funds. Green bonds reduce carbon exposure without degrading funding ratio stability.', 'investment_theory', 2023),
                ('IORP II Directive', 'The IORP II directive introduces governance and risk management requirements for occupational pension institutions across EU member states.', 'pension_regulation', 2021),
                ('BERT for Regulation', 'Fine-tuning BERT on regulatory documents improves classification accuracy. Outperforms TF-IDF baselines on pension regulation text.', 'fintech_ai', 2023),
                ('Interest Rate Risk', 'Duration mismatch between liabilities and fixed-income assets creates interest rate exposure. Liability-driven investing reduces this gap.', 'actuarial', 2022),
                ('Credit Risk ML', 'Gradient boosting for PD estimation outperforms logistic regression. Feature engineering from macroeconomic indicators improves model stability.', 'general_ml', 2023),
                ('Inflation and Pensions', 'Rising inflation affects real pension payments and liability discount rates across different regulatory regimes.', 'macroeconomics', 2023),
                ('Stochastic ALM', 'Stochastic asset-liability model for Dutch pension funds capturing equity, interest rate, and inflation risks under FTK.', 'actuarial', 2022),
                ('Dekkingsgraad Herstelplan', 'Pensioenfondsen met een dekkingsgraad onder 90% moeten een herstelplan indienen bij De Nederlandsche Bank. De beleidsdekkingsgraad bepaalt de indexatiemogelijkheden.', 'pension_regulation', 2022),
                ('Factor Investing ALM', 'Factor premia such as value, momentum, and low volatility can be harvested within ALM constraints for defined benefit pension funds.', 'investment_theory', 2022),
            ], 1)
        ]
        df = pd.DataFrame(SYNTHETIC_DATA)
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(df['abstract'].tolist(), normalize_embeddings=True)
    np.save(EMBED_PATH, embeddings)
    df.to_csv(META_PATH, index=False)
    print(f'Built and saved: {embeddings.shape}')

# Verify unit normalisation
norms = np.linalg.norm(embeddings, axis=1)
print(f'\nNorm check: min={norms.min():.6f}, max={norms.max():.6f} (should be ~1.0)')
print(f'Categories: {df["category"].value_counts().to_dict()}')

## Cosine Similarity From Scratch

Cosine similarity measures the **angle** between two vectors, not their distance:

$$\text{cos}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|}$$

Range: `−1` (opposite meaning) → `0` (orthogonal, unrelated) → `+1` (identical meaning)

**Why not Euclidean distance?** Two documents about the same topic but one is twice as long would have embeddings of different magnitudes but similar *directions*. Cosine ignores magnitude — it only cares about direction.

**The unit-sphere trick:** If both vectors are pre-normalised to unit length (`‖a‖ = ‖b‖ = 1`), then:
$$\text{cos}(\mathbf{a}, \mathbf{b}) = \mathbf{a} \cdot \mathbf{b}$$

A dot product is one matrix multiplication — much faster than computing norms at query time. This is why `semantic_search_cli.py` uses `normalize_embeddings=True` when building the index.

In [ ]:
# ── Cell 2: Cosine similarity — manual vs sklearn ────────────────────────────
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cos

def cosine_manual(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two 1-D vectors, implemented from scratch."""
    dot   = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(dot / (norm_a * norm_b))

def cosine_dot_product(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity when both vectors are pre-normalised (just a dot product)."""
    return float(np.dot(a, b))

# Test on two article embeddings
vec_a = embeddings[0]   # first article
vec_b = embeddings[1]   # second article

sim_manual = cosine_manual(vec_a, vec_b)
sim_dot    = cosine_dot_product(vec_a, vec_b)    # only valid for unit vectors
sim_sklearn = sklearn_cos(vec_a.reshape(1, -1), vec_b.reshape(1, -1))[0][0]

print(f'Article A: "{df.iloc[0]["title"]}"')
print(f'Article B: "{df.iloc[1]["title"]}"')
print()
print(f'cosine_manual()     : {sim_manual:.8f}')
print(f'dot product (normed): {sim_dot:.8f}')
print(f'sklearn cosine      : {sim_sklearn:.8f}')
print()
print('All three methods agree (for unit vectors, dot product IS cosine similarity).')

# Speed comparison
N = 10_000
t0 = time.perf_counter()
for _ in range(N): cosine_manual(vec_a, vec_b)
t_manual = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(N): cosine_dot_product(vec_a, vec_b)
t_dot = time.perf_counter() - t0

print(f'\nSpeed ({N:,} calls):')
print(f'  cosine_manual : {t_manual*1000:.1f} ms')
print(f'  dot product   : {t_dot*1000:.1f} ms  ({t_manual/t_dot:.1f}x faster)')

In [ ]:
# ── Cell 3: Brute-force nearest neighbours ────────────────────────────────────
# For a query, compute similarity against EVERY article, then sort.
# This is O(n * d) — fine for n=500, gets slow for n=1M+.

from sentence_transformers import SentenceTransformer

MODEL_NAME = 'all-MiniLM-L6-v2'
model = SentenceTransformer(MODEL_NAME)

def brute_force_search(
    query: str,
    corpus_embeddings: np.ndarray,
    df: pd.DataFrame,
    top_k: int = 5,
) -> pd.DataFrame:
    """
    Encode the query, compute cosine similarity vs. all documents,
    and return the top_k results as a DataFrame.
    """
    query_emb = model.encode([query], normalize_embeddings=True)  # shape (1, 384)

    # Matrix multiply: (n_docs, 384) @ (384, 1) = (n_docs, 1)
    # Flatten to (n_docs,) — this is cosine similarity for unit vectors
    scores = (corpus_embeddings @ query_emb.T).flatten()

    top_indices = np.argsort(scores)[::-1][:top_k]
    top_scores  = scores[top_indices]

    results = df.iloc[top_indices][['id', 'title', 'category', 'year']].copy()
    results['score'] = np.round(top_scores, 4)
    return results.reset_index(drop=True)


query = 'minimum funding requirements for pension funds under Dutch regulation'
print(f'Query: "{query}"\n')

t0 = time.perf_counter()
results = brute_force_search(query, embeddings, df, top_k=5)
elapsed_ms = (time.perf_counter() - t0) * 1000

print(results.to_string(index=False))
print(f'\nSearch time: {elapsed_ms:.1f} ms  (including encoding query)')

In [ ]:
# ── Cell 4: Keyword search vs. semantic search — head-to-head ─────────────────
# A concrete demonstration of where keyword search fails on pension domain queries.

def keyword_search(query: str, df: pd.DataFrame, top_k: int = 5) -> pd.DataFrame:
    """Simple pandas keyword search: count how many query words appear in abstract."""
    query_words = set(query.lower().split())
    # Count matched words in abstract + title
    text = (df['title'] + ' ' + df['abstract']).str.lower()
    df2 = df.copy()
    df2['kw_hits'] = text.apply(lambda t: sum(1 for w in query_words if w in t))
    top = df2.nlargest(top_k, 'kw_hits')[['id', 'title', 'category', 'year', 'kw_hits']]
    return top.reset_index(drop=True)


# Test case: Dutch term query — keyword search has zero chance
test_queries = [
    'dekkingsgraad herstelplan',          # Dutch-only query
    'LDI duration matching liabilities',  # shorthand + technical terms
    'IORP capital adequacy',              # mixed regulatory shorthand
]

for q in test_queries:
    print(f'\n{"="*65}')
    print(f'QUERY: "{q}"')
    print(f'{"="*65}')

    kw = keyword_search(q, df, top_k=3)
    sem = brute_force_search(q, embeddings, df, top_k=3)

    print(f'\nKeyword search results (by word-hit count):')
    for _, row in kw.iterrows():
        print(f'  [{int(row.kw_hits)} hits]  {row["title"]}  [{row["category"]}]')

    print(f'\nSemantic search results (by cosine similarity):')
    for _, row in sem.iterrows():
        print(f'  [{row.score:.4f}]   {row["title"]}  [{row["category"]}]')

In [ ]:
# ── Cell 5: Category filtering ────────────────────────────────────────────────
# A common real-world requirement: "find me the most relevant *regulation* articles"
# i.e. rank by semantic relevance but only within a single category.

def search_with_filter(
    query: str,
    corpus_embeddings: np.ndarray,
    df: pd.DataFrame,
    top_k: int = 5,
    category: str = None,
    min_year: int = None,
) -> pd.DataFrame:
    """
    Semantic search with optional category and year filters.
    Filtering is done BEFORE computing similarity to avoid ranking irrelevant docs.
    """
    mask = pd.Series([True] * len(df), index=df.index)
    if category:
        mask &= df['category'] == category
    if min_year:
        mask &= df['year'] >= min_year

    filtered_idx = df.index[mask].tolist()

    if not filtered_idx:
        print(f'No documents match the filters (category={category}, min_year={min_year})')
        return pd.DataFrame()

    filtered_embs = corpus_embeddings[filtered_idx]   # slice the matrix
    query_emb = model.encode([query], normalize_embeddings=True)

    scores = (filtered_embs @ query_emb.T).flatten()
    top_local  = np.argsort(scores)[::-1][:top_k]
    top_global = [filtered_idx[i] for i in top_local]
    top_scores = scores[top_local]

    results = df.iloc[top_global][['id', 'title', 'category', 'year']].copy()
    results['score'] = np.round(top_scores, 4)
    results['filter_pool'] = len(filtered_idx)
    return results.reset_index(drop=True)


print('=== Search: "solvency requirements" | No filter (all categories) ===')
r1 = search_with_filter('solvency requirements and funding ratio', embeddings, df, top_k=4)
print(r1[['score', 'title', 'category']].to_string(index=False))

print('\n=== Search: "solvency requirements" | category=pension_regulation only ===')
r2 = search_with_filter('solvency requirements and funding ratio', embeddings, df,
                         top_k=4, category='pension_regulation')
print(r2[['score', 'title', 'category', 'filter_pool']].to_string(index=False))

print('\n=== Search: "machine learning" | category=actuarial | year >= 2022 ===')
r3 = search_with_filter('machine learning mortality modelling', embeddings, df,
                         top_k=3, category='actuarial', min_year=2022)
print(r3[['score', 'title', 'category', 'year']].to_string(index=False))

In [ ]:
# ── Cell 6: Production-ready reusable search function ────────────────────────
# Clean, documented, matches the interface in semantic_search_cli.py

def semantic_search(
    query: str,
    corpus_embeddings: np.ndarray,
    df: pd.DataFrame,
    model: SentenceTransformer,
    top_k: int = 5,
    category: str = None,
    min_year: int = None,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Search the pension research corpus semantically.

    Args:
        query: Free-text question or phrase in any language.
        corpus_embeddings: Pre-computed, unit-normalised embedding matrix (n, d).
        df: Metadata DataFrame aligned row-for-row with corpus_embeddings.
        model: SentenceTransformer instance used to encode the query.
        top_k: Number of results to return.
        category: Optional category filter (e.g. 'pension_regulation').
        min_year: Optional minimum publication year filter.
        verbose: If True, print a formatted results table.

    Returns:
        DataFrame with columns: id, title, category, year, score, abstract_snippet.
    """
    # 1. Apply pre-search filters
    mask = pd.Series([True] * len(df), index=df.index)
    if category:
        if category not in df['category'].unique():
            raise ValueError(f'Unknown category: {category!r}. Valid: {list(df["category"].unique())}')
        mask &= df['category'] == category
    if min_year:
        mask &= df['year'] >= min_year

    filtered_idx = df.index[mask].tolist()
    if not filtered_idx:
        return pd.DataFrame(columns=['id', 'title', 'category', 'year', 'score', 'abstract_snippet'])

    # 2. Encode query
    query_emb = model.encode([query], normalize_embeddings=True)  # (1, d)

    # 3. Similarity scores (dot product of unit vectors = cosine similarity)
    filtered_embs = corpus_embeddings[filtered_idx]
    scores = (filtered_embs @ query_emb.T).flatten()

    # 4. Sort and select top_k
    top_local  = np.argsort(scores)[::-1][:top_k]
    top_global = [filtered_idx[i] for i in top_local]
    top_scores = scores[top_local]

    # 5. Build result DataFrame
    results = df.iloc[top_global][['id', 'title', 'category', 'year', 'abstract']].copy()
    results['score'] = np.round(top_scores, 4)
    results['abstract_snippet'] = results['abstract'].str[:120] + '...'
    results = results.drop(columns='abstract').reset_index(drop=True)

    if verbose:
        print(f"\n{'='*70}")
        print(f"Query : {query}")
        if category: print(f"Filter: category={category}")
        if min_year:  print(f"Filter: year >= {min_year}")
        print(f"Pool  : {len(filtered_idx)} articles  |  Returning top {len(results)}")
        print(f"{'='*70}")
        for i, row in results.iterrows():
            print(f"\n#{i+1}  Score={row.score:.4f}  [{row.category}]  {row.year}")
            print(f"    {row.title}")
            print(f"    {row.abstract_snippet}")
        print(f"\n{'='*70}\n")

    return results


# Demo
results = semantic_search(
    query='ESG climate risk integration in pension fund asset allocation',
    corpus_embeddings=embeddings,
    df=df,
    model=model,
    top_k=3,
    verbose=True,
)

In [ ]:
# ── Cell 7: Failure modes — where semantic search struggles ───────────────────
# Understanding failure modes is critical for production systems.

import matplotlib.pyplot as plt
import seaborn as sns

failure_queries = [
    ('Article 131 FTK',               'Numeric article references — no semantics to latch onto'),
    ('2021',                           'Year-only query — model has no date concept'),
    ('DNB circular 2023/04',           'Regulatory document codes — opaque identifiers'),
    ('short abstract < 50 words test', 'Query longer than most matching context'),
]

print('FAILURE MODE ANALYSIS — Queries where semantic search struggles\n')
score_data = []

for q, reason in failure_queries:
    query_emb = model.encode([q], normalize_embeddings=True)
    scores = (embeddings @ query_emb.T).flatten()
    top5 = np.sort(scores)[::-1][:5]
    score_data.append({
        'query': q[:35],
        'max_score': top5[0],
        'mean_top5': top5.mean(),
        'spread': top5[0] - top5[4],  # gap between rank1 and rank5
        'reason': reason,
    })
    print(f'Query   : "{q}"')
    print(f'Problem : {reason}')
    print(f'Top scores: {np.round(top5, 4).tolist()}')
    print(f'Max-min spread: {top5[0]-top5[-1]:.4f}  (small spread = model is uncertain)\n')

# Also show a GOOD query for comparison
q_good = 'pension fund coverage ratio minimum capital requirements FTK'
query_emb = model.encode([q_good], normalize_embeddings=True)
scores_good = (embeddings @ query_emb.T).flatten()
top5_good = np.sort(scores_good)[::-1][:5]
print(f'GOOD QUERY: "{q_good}"')
print(f'Top scores: {np.round(top5_good, 4).tolist()}')
print(f'Max-min spread: {top5_good[0]-top5_good[-1]:.4f}  (large spread = confident ranking)')
print()
print('KEY INSIGHT: When top-5 scores are tightly bunched, the model cannot')
print('distinguish between documents. This is a signal to fall back to keyword')
print('search, or to use a threshold (e.g. only return results with score > 0.4).')

## Key Takeaways

1. **Cosine similarity = dot product for unit vectors** — pre-normalising embeddings at index time means each query is just one matrix multiplication: `scores = corpus_embeddings @ query_emb.T`. That is a single NumPy call.

2. **Semantic search beats keyword search on jargon** — queries in Dutch, regulatory shorthand, or paraphrases all work because the embedding space captures meaning, not tokens.

3. **Category filtering happens before scoring** — slice the embedding matrix to the filtered subset, compute similarity only over that subset. Never score then filter; you waste computation and can miss relevant results due to `top_k` cutoff.

4. **Score spread is a confidence signal** — when the top-5 scores are tightly bunched (all near 0.25), the model is uncertain. Add a minimum score threshold in production (e.g. `score > 0.35`).

5. **Failure modes are predictable** — opaque identifiers (article numbers, document codes, years alone) have no semantic content. Hybrid search (BM25 + semantic) handles these; see Phase 03.

---

## Exercises

1. **Score threshold experiment.** Add a `min_score` parameter to `semantic_search()`. What threshold eliminates obviously irrelevant results without dropping good ones? Test on 5 queries.

2. **Batch queries.** Encode 10 queries at once with `model.encode(query_list)` and compute the full similarity matrix against the corpus in one operation. How does this compare to encoding queries one-by-one?

3. **Cross-lingual retrieval.** Translate 3 English queries to Dutch (use Google Translate or DeepL). Run both versions and compare top-5 results. Are the rankings stable?

4. **Keyword + semantic hybrid.** Build a function that takes a weighted sum of BM25 keyword score and cosine similarity score (`alpha * semantic + (1-alpha) * keyword`). What value of `alpha` gives the best results?

5. **Nearest neighbours for a document.** Instead of a text query, use the embedding of an existing article as the query. Which articles does the model consider most similar? Do they match the same category? Does the model find cross-category conceptual links?